In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from churn_prediction.paths import FEATURES_DIR, PROCESSED_DIR, MODELS_DIR
import pickle
import warnings
warnings.filterwarnings('ignore')

All directories created successfully!


In [2]:
print("\n1. Đọc dữ liệu...")
X = pd.read_parquet(FEATURES_DIR / 'customer_features.parquet')
Y = pd.read_parquet(PROCESSED_DIR / 'customer_labeled.parquet')

print(f"X shape: {X.shape}")
print(f"Y shape: {Y.shape}")


1. Đọc dữ liệu...
X shape: (96219, 54)
Y shape: (96096, 40)


### Hợp nhất X và Y theo customer_unique_id

In [3]:
print(X.columns.tolist())

['customer_unique_id', 'recency', 'frequency', 'monetary', 'recency_score', 'frequency_score', 'monetary_score', 'rfm_total', 'rfm_segment', 'avg_delivery_time', 'std_delivery_time', 'max_delivery_time', 'avg_estimated_delivery', 'avg_delivery_delay', 'std_delivery_delay', 'max_delivery_delay', 'avg_freight_per_order', 'total_freight_from_avg', 'total_freight_all', 'late_delivery_ratio', 'avg_review_score', 'std_review_score', 'min_review_score', 'num_comments', 'num_titles', 'avg_days_to_answer_x', 'low_review_ratio', 'num_bad_reviews', 'avg_items_per_order', 'std_items_per_order', 'weekend_purchase_ratio', 'night_purchase_ratio', 'total_orders', 'avg_gap', 'std_gap', 'max_gap', 'trend_gap', 'avg_installments', 'favorite_payment_type', 'credit_card_ratio', 'boleto_ratio', 'spending_trend', 'order_trend', 'recent_vs_old_ratio', 'avg_order_trend', 'total_comments_msg', 'total_comments_title', 'avg_days_to_answer_y', 'unique_sellers', 'comment_rate', 'customer_state', 'customer_city', 'f

In [4]:
print(X.shape)
print(X.select_dtypes(include=['object']).columns)

(96219, 54)
Index(['customer_unique_id', 'rfm_segment', 'favorite_payment_type',
       'customer_state', 'customer_city', 'favorite_category'],
      dtype='object')


In [5]:
print(X.columns)

Index(['customer_unique_id', 'recency', 'frequency', 'monetary',
       'recency_score', 'frequency_score', 'monetary_score', 'rfm_total',
       'rfm_segment', 'avg_delivery_time', 'std_delivery_time',
       'max_delivery_time', 'avg_estimated_delivery', 'avg_delivery_delay',
       'std_delivery_delay', 'max_delivery_delay', 'avg_freight_per_order',
       'total_freight_from_avg', 'total_freight_all', 'late_delivery_ratio',
       'avg_review_score', 'std_review_score', 'min_review_score',
       'num_comments', 'num_titles', 'avg_days_to_answer_x',
       'low_review_ratio', 'num_bad_reviews', 'avg_items_per_order',
       'std_items_per_order', 'weekend_purchase_ratio', 'night_purchase_ratio',
       'total_orders', 'avg_gap', 'std_gap', 'max_gap', 'trend_gap',
       'avg_installments', 'favorite_payment_type', 'credit_card_ratio',
       'boleto_ratio', 'spending_trend', 'order_trend', 'recent_vs_old_ratio',
       'avg_order_trend', 'total_comments_msg', 'total_comments_title'

In [6]:
print(Y.columns)

Index(['customer_unique_id', 'customer_id', 'city', 'state', 'zip_prefix',
       'num_orders', 'num_delivered', 'total_payment', 'total_price',
       'total_freight', 'avg_price', 'avg_freight', 'max_installments',
       'unique_products', 'unique_sellers', 'total_products',
       'avg_review_score', 'avg_days_to_answer', 'total_comments_msg',
       'total_comments_title', 'main_payment_type', 'first_purchase_date',
       'last_purchase_date', 'delivery_success_rate', 'freight_to_price_ratio',
       'avg_order_value', 'products_per_seller', 'comment_rate',
       'days_since_last_purchase', 'customer_lifetime_days',
       'days_since_first_purchase', 'avg_days_between_orders', 'churn_30',
       'churn_60', 'churn_90', 'churn_180', 'churn_365', 'is_churned',
       'churn_level', 'review_bin'],
      dtype='object')


In [7]:
print("\n2. Hợp nhất X và Y...")
y_cols = ['customer_unique_id', 'is_churned', 'churn_level']
df = X.merge(Y[y_cols], on='customer_unique_id', how='inner')

print(f"Hợp nhất thành công: {df.shape}")
print(f"Churn rate trong tập hợp nhất: {df['is_churned'].mean()*100:.2f}%")


2. Hợp nhất X và Y...
Hợp nhất thành công: (96219, 56)
Churn rate trong tập hợp nhất: 89.94%


### 3. Loại bỏ các cột không cần thiết

In [8]:
print("\n3. Loại bỏ cột không cần thiết...")
# Giữ lại customer_unique_id để trace, nhưng sẽ không dùng cho model
# Các cột text không có giá trị cho mô hình số sẽ được mã hóa sau
drop_cols = ['customer_unique_id', 'rfm_segment', 'favorite_category', 
             'favorite_payment_type', 'customer_city', 'customer_state', 'recency','frequency',
             'monetary','recency_score','frequency_score','monetary_score','rfm_total','total_orders']
# Chỉ drop nếu tồn tại
for col in drop_cols:
    if col in df.columns:
        df = df.drop(col, axis=1)
print(f"   Số cột sau khi drop: {df.shape[1]}")


3. Loại bỏ cột không cần thiết...
   Số cột sau khi drop: 42


In [9]:
print (df.columns)

Index(['avg_delivery_time', 'std_delivery_time', 'max_delivery_time',
       'avg_estimated_delivery', 'avg_delivery_delay', 'std_delivery_delay',
       'max_delivery_delay', 'avg_freight_per_order', 'total_freight_from_avg',
       'total_freight_all', 'late_delivery_ratio', 'avg_review_score',
       'std_review_score', 'min_review_score', 'num_comments', 'num_titles',
       'avg_days_to_answer_x', 'low_review_ratio', 'num_bad_reviews',
       'avg_items_per_order', 'std_items_per_order', 'weekend_purchase_ratio',
       'night_purchase_ratio', 'avg_gap', 'std_gap', 'max_gap', 'trend_gap',
       'avg_installments', 'credit_card_ratio', 'boleto_ratio',
       'spending_trend', 'order_trend', 'recent_vs_old_ratio',
       'avg_order_trend', 'total_comments_msg', 'total_comments_title',
       'avg_days_to_answer_y', 'unique_sellers', 'comment_rate',
       'num_categories_bought', 'is_churned', 'churn_level'],
      dtype='object')


### 4. Xử lý missing values

In [10]:
print("\n4. Xử lý missing values...")
missing_before = df.isnull().sum().sum()
df = df.fillna(0)  
print(f"Missing trước: {missing_before}, sau: {df.isnull().sum().sum()}")


4. Xử lý missing values...
Missing trước: 0, sau: 0


### 5. Mã hóa biến phân loại (categorical)

In [12]:
print("\n5. Mã hóa categorical features...")
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
# Loại bỏ các cột target không cần encode
target_cols = ['is_churned', 'churn_level']
categorical_cols = [c for c in categorical_cols if c not in target_cols]

encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    encoders[col] = le
    print(f"Encoded {col} -> {len(le.classes_)} categories")
# Lưu encoders để dùng sau
with open(MODELS_DIR / 'label_encoders.pkl', 'wb') as f:
    pickle.dump(encoders, f)


5. Mã hóa categorical features...


### 6. Feature Selection (chọn đặc trưng quan trọng)

In [13]:
print("\n6. Feature Selection...")
# Chọn target là is_churned
X_features = df.drop(columns=target_cols, axis=1)
y_target = df['is_churned']

# Sử dụng Mutual Information để đánh giá tầm quan trọng
selector = SelectKBest(score_func=mutual_info_classif, k=30)  # chọn top 30 features
X_selected = selector.fit_transform(X_features, y_target)

# Lấy tên các feature được chọn
selected_mask = selector.get_support()
selected_features = X_features.columns[selected_mask].tolist()
print(f"   Top 30 features được chọn:")
for i, feat in enumerate(selected_features[:10], 1):
    print(f"{i}. {feat}")
if len(selected_features) > 10:
    print(f"... và {len(selected_features)-10} features khác")

# Lưu selector
with open(MODELS_DIR / 'feature_selector.pkl', 'wb') as f:
    pickle.dump(selector, f)


6. Feature Selection...
   Top 30 features được chọn:
1. avg_delivery_time
2. max_delivery_time
3. avg_estimated_delivery
4. avg_delivery_delay
5. max_delivery_delay
6. avg_freight_per_order
7. total_freight_from_avg
8. total_freight_all
9. late_delivery_ratio
10. avg_review_score
... và 20 features khác


In [14]:
for i, feat in enumerate(selected_features[:30], 1):
    print(f"{i}. {feat}")

1. avg_delivery_time
2. max_delivery_time
3. avg_estimated_delivery
4. avg_delivery_delay
5. max_delivery_delay
6. avg_freight_per_order
7. total_freight_from_avg
8. total_freight_all
9. late_delivery_ratio
10. avg_review_score
11. min_review_score
12. num_comments
13. num_titles
14. avg_days_to_answer_x
15. low_review_ratio
16. avg_items_per_order
17. weekend_purchase_ratio
18. night_purchase_ratio
19. max_gap
20. trend_gap
21. avg_installments
22. credit_card_ratio
23. boleto_ratio
24. avg_order_trend
25. total_comments_msg
26. total_comments_title
27. avg_days_to_answer_y
28. unique_sellers
29. comment_rate
30. num_categories_bought


### 7. Chia dữ liệu (train/validation/test)

In [15]:
print("\n7. Chia dữ liệu")
# Tỷ lệ: 70% train, 15% validation, 15% test
X_train, X_temp, y_train, y_temp = train_test_split(
    X_features[selected_features], y_target, test_size=0.3, random_state=42, stratify=y_target
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print(f"   Train: {X_train.shape[0]} samples")
print(f"   Validation: {X_val.shape[0]} samples")
print(f"   Test: {X_test.shape[0]} samples")
print(f"   Churn rate - Train: {y_train.mean()*100:.2f}%")
print(f"   Churn rate - Val: {y_val.mean()*100:.2f}%")
print(f"   Churn rate - Test: {y_test.mean()*100:.2f}%")


7. Chia dữ liệu
   Train: 67353 samples
   Validation: 14433 samples
   Test: 14433 samples
   Churn rate - Train: 89.94%
   Churn rate - Val: 89.94%
   Churn rate - Test: 89.94%


### 8. Chuẩn hóa dữ liệu (StandardScaler)

In [17]:
print("\n8. Chuẩn hóa dữ liệu...")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# Lưu scaler
with open(MODELS_DIR / 'scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)


8. Chuẩn hóa dữ liệu...


### 9. Lưu các tập dữ liệu đã xử lý

In [18]:
print("\n9. Lưu dữ liệu đã xử lý")
# Lưu dạng numpy arrays để dễ dùng cho modeling
np.save(MODELS_DIR / 'X_train.npy', X_train_scaled)
np.save(MODELS_DIR / 'X_val.npy', X_val_scaled)
np.save(MODELS_DIR / 'X_test.npy', X_test_scaled)
np.save(MODELS_DIR / 'y_train.npy', y_train.values)
np.save(MODELS_DIR / 'y_val.npy', y_val.values)
np.save(MODELS_DIR / 'y_test.npy', y_test.values)

# Cũng lưu dạng pandas để tiện kiểm tra
pd.DataFrame(X_train_scaled, columns=selected_features).to_parquet(MODELS_DIR / 'X_train.parquet')
pd.DataFrame(y_train).to_parquet(MODELS_DIR / 'y_train.parquet')


9. Lưu dữ liệu đã xử lý


### 10. Tổng kết pipeline

In [19]:
print("DATA PIPELINE HOÀN TẤT!")

print("\nCác file đã tạo:")
print("- label_encoders.pkl: để mã hóa categorical features khi dự đoán mới")
print("- feature_selector.pkl: để chọn đúng features cho dữ liệu mới")
print("- scaler.pkl: để chuẩn hóa dữ liệu mới")
print("- X_train.npy, X_val.npy, X_test.npy: features đã chuẩn hóa")
print("- y_train.npy, y_val.npy, y_test.npy: labels")
print("\nSẵn sàng cho huấn luyện mô hình và tích hợp multi-agent!")

DATA PIPELINE HOÀN TẤT!

Các file đã tạo:
- label_encoders.pkl: để mã hóa categorical features khi dự đoán mới
- feature_selector.pkl: để chọn đúng features cho dữ liệu mới
- scaler.pkl: để chuẩn hóa dữ liệu mới
- X_train.npy, X_val.npy, X_test.npy: features đã chuẩn hóa
- y_train.npy, y_val.npy, y_test.npy: labels

Sẵn sàng cho huấn luyện mô hình và tích hợp multi-agent!
